# Ejemplo $B^{\pm}\to J/\psi K^{\pm}$ con Uproot

Versión modular compatible con Windows. Los modelos y la lógica de ajuste están en `ecfm_uproot/fitting.py`, de modo que esta libreta se concentra en comparar los resultados.

In [ ]:
from pathlib import Path
import sys

# Funciona si VS Code inicia el kernel en la raíz del repositorio o en Dec_Bpm.
module_candidates = (
    Path.cwd(),
    Path.cwd() / "Pruebas" / "Dec_Bpm",
)
for candidate in module_candidates:
    if (candidate / "ecfm_uproot").is_dir():
        analysis_dir = candidate.resolve()
        break
else:
    raise FileNotFoundError(
        "No se encontró la carpeta ecfm_uproot. Copia el paquete junto a los notebooks."
    )

if str(analysis_dir) not in sys.path:
    sys.path.insert(0, str(analysis_dir))

print("Intérprete de Python:", sys.executable)

import numpy as np

from ecfm_uproot import load_arrays
from ecfm_uproot.fitting import (
    breit_wigner,
    double_gaussian,
    double_gaussian_plus_exponential,
    exponential_background,
    fit_and_plot,
    gaussian,
    gaussian_plus_exponential,
    gaussian_plus_polynomial_1,
    landau_like,
    polynomial_1,
    polynomial_2,
    polynomial_3,
)
from ecfm_uproot.plotting import plot_hist1d

arrays = load_arrays(["Bplus_M", "J_psi_1S_M"])
b_mass = arrays["Bplus_M"]
jpsi_mass = arrays["J_psi_1S_M"]

hist_counts, _ = np.histogram(b_mass, bins=200, range=(4600, 6000))
peak_count = max(hist_counts.max(), 1)
background_count = max(np.median(hist_counts), 1)

print(f"Candidatos cargados: {len(b_mass):,}")


## Histogramas de masa

In [ ]:
plot_hist1d(
    b_mass,
    bins=200,
    value_range=(4600, 6000),
    title="Masa del candidato B",
    xlabel=r"$m_{\mu^+\mu^-K^{\pm}}$ [MeV/$c^2$]",
    filename="ej01_bmass_uproot.png",
)

plot_hist1d(
    jpsi_mass,
    bins=200,
    value_range=(2950, 3250),
    title=r"Masa del candidato $J/\psi$",
    xlabel=r"$m_{\mu^+\mu^-}$ [MeV/$c^2$]",
    filename="ej01_jpsimass_uproot.png",
)


## Ajustes polinomiales

In [ ]:
fit_B_plin = fit_and_plot(
    b_mass,
    model_name="pol1",
    model=polynomial_1,
    initial_parameters=[background_count, 0],
    bounds=([-np.inf, -np.inf], [np.inf, np.inf]),
    parameter_names=["c0", "c1"],
    bins=200,
    histogram_range=(4600, 6000),
    fit_range=(5000, 5600),
    title="Ajuste polinomial lineal",
    xlabel=r"$m_{\mu^+\mu^-K^{\pm}}$ [MeV/$c^2$]",
    filename="ej01_fit_plin_bmass_uproot.png",
)


In [ ]:
fit_B_pcuad = fit_and_plot(
    b_mass,
    model_name="pol2",
    model=polynomial_2,
    initial_parameters=[peak_count, 0, 0],
    bounds=([-np.inf] * 3, [np.inf] * 3),
    parameter_names=["c0", "c1", "c2"],
    bins=200,
    histogram_range=(4600, 6000),
    fit_range=(5200, 5360),
    title="Ajuste polinomial cuadrático",
    xlabel=r"$m_{\mu^+\mu^-K^{\pm}}$ [MeV/$c^2$]",
    filename="ej01_fit_pcuad_bmass_uproot.png",
)


In [ ]:
fit_B_pcub = fit_and_plot(
    b_mass,
    model_name="pol3",
    model=polynomial_3,
    initial_parameters=[background_count, 0, 0, 0],
    bounds=([-np.inf] * 4, [np.inf] * 4),
    parameter_names=["c0", "c1", "c2", "c3"],
    bins=200,
    histogram_range=(4600, 6000),
    fit_range=(5000, 5600),
    title="Ajuste polinomial cúbico",
    xlabel=r"$m_{\mu^+\mu^-K^{\pm}}$ [MeV/$c^2$]",
    filename="ej01_fit_pcub_bmass_uproot.png",
)


## Ajuste exponencial

In [ ]:
fit_B_exp = fit_and_plot(
    b_mass,
    model_name="exponential",
    model=exponential_background,
    initial_parameters=[background_count, -0.001],
    bounds=([0, -0.05], [np.inf, 0.05]),
    parameter_names=["amplitud", "pendiente"],
    bins=200,
    histogram_range=(4600, 6000),
    fit_range=(5000, 5600),
    title="Ajuste exponencial",
    xlabel=r"$m_{\mu^+\mu^-K^{\pm}}$ [MeV/$c^2$]",
    filename="ej01_fit_exp_bmass_uproot.png",
)


## Ajuste tipo Landau

`landau_like` usa la aproximación de Moyal a la forma asimétrica de Landau.

In [ ]:
fit_B_landau = fit_and_plot(
    b_mass,
    model_name="landau_like",
    model=landau_like,
    initial_parameters=[peak_count, 5280, 30],
    bounds=([0, 5000, 1], [np.inf, 5600, 500]),
    parameter_names=["amplitud", "valor_mas_probable", "anchura"],
    bins=200,
    histogram_range=(4600, 6000),
    fit_range=(5000, 5600),
    title="Ajuste tipo Landau (aproximación de Moyal)",
    xlabel=r"$m_{\mu^+\mu^-K^{\pm}}$ [MeV/$c^2$]",
    filename="ej01_fit_landau_bmass_uproot.png",
)


## Ajuste gaussiano

In [ ]:
fit_B_gaus = fit_and_plot(
    b_mass,
    model_name="gaussian",
    model=gaussian,
    initial_parameters=[peak_count, 5280, 20],
    bounds=([0, 5000, 1], [np.inf, 5600, 300]),
    parameter_names=["amplitud", "media", "sigma"],
    bins=200,
    histogram_range=(4600, 6000),
    fit_range=(5000, 5600),
    title="Ajuste gaussiano",
    xlabel=r"$m_{\mu^+\mu^-K^{\pm}}$ [MeV/$c^2$]",
    filename="ej01_fit_gaus_bmass_uproot.png",
)


## Gaussiana + fondo lineal

In [ ]:
fit_R_gauspol1 = fit_and_plot(
    b_mass,
    model_name="gaussian_plus_pol1",
    model=gaussian_plus_polynomial_1,
    initial_parameters=[peak_count, 5280, 20, background_count, 0],
    bounds=([0, 5150, 1, -np.inf, -np.inf], [np.inf, 5400, 200, np.inf, np.inf]),
    parameter_names=["amplitud_gauss", "media", "sigma", "c0", "c1"],
    bins=200,
    histogram_range=(4600, 6000),
    fit_range=(5150, 5400),
    title="Gaussiana + fondo lineal",
    xlabel=r"$m_{\mu^+\mu^-K^{\pm}}$ [MeV/$c^2$]",
    filename="ej01_fit_gp1_bmass_uproot.png",
)


## Gaussiana + fondo exponencial

In [ ]:
fit_R_gausexp = fit_and_plot(
    b_mass,
    model_name="gaussian_plus_exponential",
    model=gaussian_plus_exponential,
    initial_parameters=[peak_count, 5280, 20, background_count, -0.001],
    bounds=([0, 5150, 1, 0, -0.05], [np.inf, 5400, 200, np.inf, 0.05]),
    parameter_names=["amplitud_gauss", "media", "sigma", "amplitud_fondo", "pendiente"],
    bins=200,
    histogram_range=(4600, 6000),
    fit_range=(5150, 5400),
    title="Gaussiana + fondo exponencial",
    xlabel=r"$m_{\mu^+\mu^-K^{\pm}}$ [MeV/$c^2$]",
    filename="ej01_fit_gex_bmass_uproot.png",
)


## Doble gaussiana

In [ ]:
fit_R_gaus2 = fit_and_plot(
    b_mass,
    model_name="double_gaussian",
    model=double_gaussian,
    initial_parameters=[0.7 * peak_count, 5280, 15, 0.3 * peak_count, 5280, 35],
    bounds=([0, 5240, 1, 0, 5240, 25], [np.inf, 5320, 40, np.inf, 5320, 150]),
    parameter_names=["a1", "media1", "sigma1", "a2", "media2", "sigma2"],
    bins=200,
    histogram_range=(4600, 6000),
    fit_range=(5150, 5400),
    title="Doble gaussiana",
    xlabel=r"$m_{\mu^+\mu^-K^{\pm}}$ [MeV/$c^2$]",
    filename="ej01_fit_g2_bmass_uproot.png",
)


## Doble gaussiana + fondo exponencial

In [ ]:
double_gaussian_bounds = (
    [0, 5240, 1, 0, 5240, 25, 0, -0.05],
    [np.inf, 5320, 40, np.inf, 5320, 150, np.inf, 0.05],
)

fit_R_gaus2expo = fit_and_plot(
    b_mass,
    model_name="double_gaussian_plus_exponential",
    model=double_gaussian_plus_exponential,
    initial_parameters=[
        0.7 * peak_count, 5280, 15,
        0.3 * peak_count, 5280, 35,
        background_count, -0.001,
    ],
    bounds=double_gaussian_bounds,
    parameter_names=["a1", "media1", "sigma1", "a2", "media2", "sigma2", "fondo", "pendiente"],
    bins=200,
    histogram_range=(4600, 6000),
    fit_range=(5150, 5400),
    title="Doble gaussiana + fondo exponencial",
    xlabel=r"$m_{\mu^+\mu^-K^{\pm}}$ [MeV/$c^2$]",
    filename="ej01_fit_g2e_bmass_uproot.png",
)


### Segunda inicialización del fondo

In [ ]:
fit2_R_gaus2expo = fit_and_plot(
    b_mass,
    model_name="double_gaussian_plus_exponential_2",
    model=double_gaussian_plus_exponential,
    initial_parameters=[
        0.7 * peak_count, 5280, 15,
        0.3 * peak_count, 5280, 35,
        background_count, -0.0001,
    ],
    bounds=double_gaussian_bounds,
    parameter_names=["a1", "media1", "sigma1", "a2", "media2", "sigma2", "fondo", "pendiente"],
    bins=200,
    histogram_range=(4600, 6000),
    fit_range=(5150, 5400),
    title="Doble gaussiana + fondo exponencial (segunda inicialización)",
    xlabel=r"$m_{\mu^+\mu^-K^{\pm}}$ [MeV/$c^2$]",
    filename="ej01_fit2_g2e_bmass_uproot.png",
)


## Breit–Wigner

In [ ]:
fit_R_bw = fit_and_plot(
    b_mass,
    model_name="breit_wigner",
    model=breit_wigner,
    initial_parameters=[peak_count, 5280, 20],
    bounds=([0, 5150, 1], [np.inf, 5400, 300]),
    parameter_names=["amplitud", "media", "gamma"],
    bins=200,
    histogram_range=(4600, 6000),
    fit_range=(5150, 5400),
    title="Ajuste Breit–Wigner",
    xlabel=r"$m_{\mu^+\mu^-K^{\pm}}$ [MeV/$c^2$]",
    filename="ej01_fit_bw_bmass_uproot.png",
)
